# 04. HIN 그래프 빌더

**소스**: `data/processed/hin/최종/` (product_final_keywords, product_ip_mapping, keyword_review, product_promo_keywords)  
**출력**: `data/processed/hin/최종/` — 노드 4종 + 엣지 5종 parquet

| 노드 | 파일 |
|------|------|
| 제품 | `product_nodes.parquet` |
| IP | `ip_nodes.parquet` |
| 키워드 | `keyword_nodes.parquet` |
| 프로모션 | `promo_nodes.parquet` |

| 엣지 | 파일 |
|------|------|
| 제품→키워드 | `product_keyword_edges.parquet` |
| IP→키워드 | `ip_keyword_edges.parquet` |
| 트렌드→키워드 | `trend_keyword_edges.parquet` |
| 제품→IP | `product_ip_edges.parquet` |
| 제품→프로모션 | `product_promo_edges.parquet` |

## Phase 0. 환경 설정 & 데이터 로드

In [95]:
import re
import ast
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR  = Path("../..").resolve()
PROC      = BASE_DIR / "data" / "processed"
HIN_DIR   = PROC / "hin"
HIN_DIR.mkdir(exist_ok=True)

def norm_id(x):
    try: return str(int(float(x)))
    except: return str(x)

def parse_kw(v):
    """쉼표 구분 문자열 또는 리스트 → 파이썬 리스트"""
    if isinstance(v, (list, np.ndarray)):
        return [str(k).strip() for k in v if str(k).strip()]
    if pd.isna(v) or str(v).strip() == "":
        return []
    return [k.strip() for k in str(v).split(",") if k.strip()]

def to_list(v):
    if isinstance(v, (list, np.ndarray)):
        return list(v)
    return []

# ── 공통 소스 ─────────────────────────────────────────────────────
df_pos    = pd.read_parquet(PROC / "pos_product_features.parquet")
df_label  = pd.read_csv(PROC / "npd_success_labels.csv", encoding="utf-8-sig")
df_insta  = pd.read_parquet(PROC / "instagram_engagement_with_keywords.parquet")
df_bridge = pd.read_parquet(PROC / "seven_eleven_product_master.parquet")
df_trend  = pd.read_parquet(PROC / "trend_keywords.parquet")

# ── HIN 폴더 파일 ─────────────────────────────────────────────────
df_pfk   = pd.read_csv(HIN_DIR / "product_final_keywords.csv", encoding="utf-8-sig")
df_pfk["ITEM_CD"]      = df_pfk["ITEM_CD"].apply(norm_id)
df_pfk["키워드_list"]  = df_pfk["키워드"].apply(parse_kw)

xl_ip    = pd.ExcelFile(HIN_DIR / "product_ip_mapping.xlsx")
df_ip    = xl_ip.parse("IP별_키워드")
df_ip["키워드_final"] = df_ip["키워드_final"].apply(parse_kw)

df_promo = pd.read_csv(HIN_DIR / "product_promo_keywords.csv", encoding="utf-8-sig")
df_promo["ITEM_CD"] = df_promo["ITEM_CD"].apply(norm_id)

print("로드 완료")
print(f"  pos:    {len(df_pos)}행   /  label:  {len(df_label)}행")
print(f"  insta:  {len(df_insta)}행  /  bridge: {len(df_bridge)}행")
print(f"  pfk:    {len(df_pfk)}행   /  ip:     {len(df_ip)}행")
print(f"  promo:  {len(df_promo)}행  /  trend:  {len(df_trend)}행")

로드 완료
  pos:    2858행   /  label:  5143행
  insta:  4249행  /  bridge: 10586행
  pfk:    5051행   /  ip:     276행
  promo:  1475행  /  trend:  747행


## Phase 1. 제품 노드

**스키마**: `ITEM_CD`, `ITEM_NM`, `has_promo_30d`, `성공여부`, `첫_등장일`, `인스타_언급횟수`, `인스타_언급일자`, `키워드_final`

In [96]:
# ── 1-2. 인스타 집계: 메타 정보만 (언급횟수 · 언급일자 · 첫등장일) ─
# 키워드는 pfk(product_final_keywords.csv)에서 직접 가져오므로 여기선 메타만 집계.

# ── 세븐일레븐: 브릿지 경유 ITEM_CD 해소 ─────────────────────────
bridge_norm_map = (
    df_bridge[df_bridge["인스타_정규화명"].notna()]
    .assign(ITEM_CD=lambda d: d["ITEM_CD"].apply(norm_id))
    [["인스타_정규화명", "ITEM_CD"]]
    .drop_duplicates("인스타_정규화명")
    .set_index("인스타_정규화명")["ITEM_CD"]
    .to_dict()
)

insta_7 = df_insta[df_insta["편의점명"] == "세븐일레븐"].copy()
insta_7["_item_cd"] = (
    insta_7["ITEM_CD"]
    .apply(lambda x: norm_id(x) if pd.notna(x) else None)
    .fillna(insta_7["정규화명"].map(bridge_norm_map))
)

insta_meta_7 = (
    insta_7[insta_7["_item_cd"].notna()]
    .assign(_date=lambda d: pd.to_datetime(d["언급일"], errors="coerce"))
    .groupby("_item_cd")
    .agg(
        인스타_언급횟수 = ("_item_cd", "count"),
        인스타_언급일자 = ("_date", lambda s: sorted(set(str(d.date()) for d in s if pd.notna(d)))),
        인스타_첫등장일 = ("_date", "min"),
    )
    .reset_index()
    .rename(columns={"_item_cd": "ITEM_CD"})
)

# ── CU/GS25: 합성 ITEM_CD = {편의점명}_{정규화명} ────────────────
insta_cg = df_insta[df_insta["편의점명"].isin(["CU", "GS25"])].copy()
insta_cg["_item_cd"] = insta_cg["편의점명"] + "_" + insta_cg["정규화명"]

insta_meta_cg = (
    insta_cg[insta_cg["_item_cd"].notna()]
    .assign(_date=lambda d: pd.to_datetime(d["언급일"], errors="coerce"))
    .groupby("_item_cd")
    .agg(
        인스타_언급횟수 = ("_item_cd", "count"),
        인스타_언급일자 = ("_date", lambda s: sorted(set(str(d.date()) for d in s if pd.notna(d)))),
        인스타_첫등장일 = ("_date", "min"),
    )
    .reset_index()
    .rename(columns={"_item_cd": "ITEM_CD"})
)

insta_meta = pd.concat([insta_meta_7, insta_meta_cg], ignore_index=True)

print(f"브릿지 커버: {len(bridge_norm_map)}개 정규화명 → ITEM_CD")
print(f"인스타 메타 집계 — 세븐일레븐: {len(insta_meta_7)}개 / CU+GS25: {len(insta_meta_cg)}개 / 합계: {len(insta_meta)}개")

브릿지 커버: 659개 정규화명 → ITEM_CD
인스타 메타 집계 — 세븐일레븐: 659개 / CU+GS25: 2147개 / 합계: 2806개


In [97]:
# ── 1-3. 제품 노드 조립 ──────────────────────────────────────────
df_pos["ITEM_CD"]   = df_pos["ITEM_CD"].apply(norm_id)
df_label["ITEM_CD"] = df_label["ITEM_CD"].apply(norm_id)

# 기본 테이블: 성공 라벨 기준 전 제품
prod = (
    df_label[["ITEM_CD", "상품명", "성공여부", "편의점명", "소스"]]
    .rename(columns={"상품명": "ITEM_NM", "소스": "성공_소스"})
    .copy()
    .merge(
        df_pos[["ITEM_CD", "첫판매일"]].rename(columns={"첫판매일": "_pos_date"}),
        on="ITEM_CD", how="left",
    )
)

# 인스타 메타 병합
prod = prod.merge(
    insta_meta.rename(columns={"인스타_첫등장일": "_insta_date"}),
    on="ITEM_CD", how="left",
)

# pfk에서 키워드_final 병합
prod = prod.merge(df_pfk[["ITEM_CD", "키워드_list"]], on="ITEM_CD", how="left")
prod["키워드_final"] = prod["키워드_list"].apply(
    lambda v: v if isinstance(v, list) else []
)

# 첫_등장일: POS 우선 → 인스타 fallback
prod["첫_등장일"] = pd.to_datetime(prod["_pos_date"]).fillna(prod["_insta_date"])

# insta_mention_30d: 첫_등장일 + 30일 이내 인스타 언급 횟수
def _count_30d(row):
    dates = row["인스타_언급일자"]
    first = row["첫_등장일"]
    if not isinstance(dates, (list, np.ndarray)) or len(dates) == 0 or pd.isna(first):
        return 0
    cutoff = pd.Timestamp(first) + pd.Timedelta(days=30)
    return int(sum(pd.Timestamp(d) <= cutoff for d in dates))

prod["insta_mention_30d"] = prod.apply(_count_30d, axis=1)

product_nodes = prod[[
    "ITEM_CD", "ITEM_NM", "편의점명", "성공여부", "성공_소스",
    "첫_등장일", "인스타_언급횟수", "인스타_언급일자", "insta_mention_30d", "키워드_final",
]].copy()

product_nodes.to_parquet(HIN_DIR / "product_nodes.parquet", index=False)
print(f"product_nodes: {len(product_nodes)}행 → {HIN_DIR / 'product_nodes.parquet'}")
print()
print("편의점별 제품 수:")
print(product_nodes["편의점명"].value_counts().to_string())
print()
print(f"키워드 있는 제품: {(product_nodes['키워드_final'].apply(len) > 0).sum()}개 / {len(product_nodes)}개")
product_nodes.head(3)

product_nodes: 5161행 → C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\hin\product_nodes.parquet

편의점별 제품 수:
편의점명
세븐일레븐    2996
CU       1162
GS25     1003

키워드 있는 제품: 5051개 / 5161개


,ITEM_CD,ITEM_NM,편의점명,성공여부,성공_소스,첫_등장일,인스타_언급횟수,인스타_언급일자,insta_mention_30d,키워드_final
0,53047,존슨빌)폴리쉬소시지108g,세븐일레븐,실패,POS,2025-02-23,NaN,NaN,0,"[야식, 간식, 소시지, 존슨빌, 폴리쉬]"
1,53197,요기요)만쿠만구치킨,세븐일레븐,실패,POS,2025-04-14,NaN,NaN,0,"[야식, 간식, 편의점, 치킨, 닭다리, 바삭, 가성비, 만쿠만구]"
2,14173,예약)한끼연구소 간장불고기정식,세븐일레븐,실패,POS,2025-05-30,NaN,NaN,0,"[도시락, 점심, 반찬, 직장, 한식, 밥, 고기, 간장, 불고기, 정식]"


## Phase 2. IP 노드

In [98]:
ip_nodes = df_ip[["ip_name", "키워드_final"]].copy()
ip_nodes.to_parquet(HIN_DIR / "ip_nodes.parquet", index=False)
print(f"ip_nodes: {len(ip_nodes)}행 → {HIN_DIR / 'ip_nodes.parquet'}")
ip_nodes.head(3)

ip_nodes: 276행 → C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\hin\ip_nodes.parquet


,ip_name,키워드_final
0,APEC정상회의공식협찬사,"[공식, 글로벌, 한정, 마케팅]"
1,BHC,"[치킨, 달콤, 간장, 마늘, 배달]"
2,EBS,"[교육, 신뢰, 공부, 공익, 다큐멘터리]"


## Phase 3. 키워드 노드

In [99]:
# 키워드 vocab: pfk(제품) ∪ ip Sheet2(IP) 합집합
prod_kw_vocab = {k for lst in df_pfk["키워드_list"] for k in lst}
ip_kw_vocab   = {k for lst in df_ip["키워드_final"] for k in lst}
all_kw_vocab  = prod_kw_vocab | ip_kw_vocab

print(f"제품 키워드: {len(prod_kw_vocab)}개 / IP 키워드: {len(ip_kw_vocab)}개 / 합집합: {len(all_kw_vocab)}개")

# 트렌드 키워드셋 + 속성 매핑
trend_kw_set = set(df_trend["트렌드_키워드"].astype(str))
trend_attr   = df_trend.set_index("트렌드_키워드")["추출_속성_final"].to_dict()

# 키워드별 인스타 첫 등장일 (트렌드 키워드 전용)
insta_kw_dates = []
for _, row in df_insta.iterrows():
    kws  = to_list(row.get("키워드_final"))
    date = row.get("언급일")
    for kw in kws:
        insta_kw_dates.append({"keyword": str(kw), "언급일": date})

kw_first_date = (
    pd.DataFrame(insta_kw_dates)
    .assign(언급일=lambda d: pd.to_datetime(d["언급일"], errors="coerce"))
    .groupby("keyword")["언급일"].min()
    .reset_index()
    .rename(columns={"언급일": "인스타_첫_등장일"})
)

# 키워드 노드 조립
kw_nodes = pd.DataFrame({"keyword": sorted(all_kw_vocab)})
kw_nodes["is_trend_keyword"] = kw_nodes["keyword"].isin(trend_kw_set)
kw_nodes["추출_속성"] = kw_nodes["keyword"].map(trend_attr).apply(
    lambda v: v if isinstance(v, list) else []
)
kw_nodes = kw_nodes.merge(kw_first_date, on="keyword", how="left")
kw_nodes.loc[~kw_nodes["is_trend_keyword"], "인스타_첫_등장일"] = None

kw_nodes.to_parquet(HIN_DIR / "keyword_nodes.parquet", index=False)
print(f"keyword_nodes: {len(kw_nodes)}행 (트렌드: {kw_nodes['is_trend_keyword'].sum()}개) → saved")
kw_nodes.head(3)

제품 키워드: 4548개 / IP 키워드: 756개 / 합집합: 4940개
keyword_nodes: 4940행 (트렌드: 402개) → saved


,keyword,is_trend_keyword,추출_속성,인스타_첫_등장일
0,'귀여움',False,[],NaT
1,'목장',False,[],NaT
2,'신선',False,[],NaT


## Phase 3.5. 키워드 EDA — 노이즈 후보 확인

키워드별 등장 제품 수 / IP 수를 집계하여 저빈도 노이즈 후보를 확인합니다.
제거 목록은 다음 Phase 3.6 셀의 `NOISE_REMOVE`에 직접 기입합니다.

In [100]:
# ── 키워드별 등장 제품 수 + IP 수 집계 ───────────────────────────
kw_prod_count = (
    product_nodes[["ITEM_CD", "ITEM_NM", "키워드_final"]]
    .explode("키워드_final")
    .rename(columns={"키워드_final": "keyword"})
    .dropna(subset=["keyword"])
    .assign(keyword=lambda d: d["keyword"].astype(str))
)

# 제품 수
kw_prod_n = (
    kw_prod_count.groupby("keyword")["ITEM_CD"].nunique()
    .reset_index().rename(columns={"ITEM_CD": "제품수"})
)

# 제품명 예시 3개
kw_prod_examples = (
    kw_prod_count.drop_duplicates(["keyword", "ITEM_NM"])
    .groupby("keyword")["ITEM_NM"]
    .apply(lambda s: " / ".join(s.head(3).tolist()))
    .reset_index().rename(columns={"ITEM_NM": "제품명_예시"})
)

kw_ip_count = (
    ip_nodes[["ip_name", "키워드_final"]]
    .explode("키워드_final")
    .rename(columns={"키워드_final": "keyword"})
    .dropna(subset=["keyword"])
    .assign(keyword=lambda d: d["keyword"].astype(str))
    .groupby("keyword")["ip_name"].nunique()
    .reset_index().rename(columns={"ip_name": "IP수"})
)

kw_eda = (
    kw_nodes[["keyword"]]
    .merge(kw_prod_n,        on="keyword", how="left")
    .merge(kw_prod_examples, on="keyword", how="left")
    .merge(kw_ip_count,      on="keyword", how="left")
)
kw_eda["제품수"]   = kw_eda["제품수"].fillna(0).astype(int)
kw_eda["IP수"]     = kw_eda["IP수"].fillna(0).astype(int)
kw_eda["총등장수"] = kw_eda["제품수"] + kw_eda["IP수"]
kw_eda["제품명_예시"] = kw_eda["제품명_예시"].fillna("")

# ── CSV 저장 ─────────────────────────────────────────────────────
eda_csv = HIN_DIR / "keyword_eda.csv"
kw_eda.sort_values("총등장수")[["keyword","제품수","IP수","총등장수","제품명_예시"]].to_csv(
    eda_csv, index=False, encoding="utf-8-sig"
)

print(f"전체 키워드: {len(kw_eda)}개  →  {eda_csv}")
print()
print("총등장수 분포 (하위 구간):")
print(kw_eda["총등장수"].value_counts().sort_index().head(10).to_string())
print()
print(f"총등장수=0: {(kw_eda['총등장수']==0).sum()}개")
print(f"총등장수=1: {(kw_eda['총등장수']==1).sum()}개")
print(f"총등장수=2: {(kw_eda['총등장수']==2).sum()}개")
print()
print("총등장수 하위 30개:")
display(
    kw_eda.sort_values("총등장수")
    .head(30)[["keyword","제품수","IP수","총등장수","제품명_예시"]]
    .reset_index(drop=True)
)

전체 키워드: 4940개  →  C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\hin\keyword_eda.csv

총등장수 분포 (하위 구간):
총등장수
1     2590
2      734
3      364
4      208
5      151
6      105
7       86
8       67
9       49
10      49

총등장수=0: 0개
총등장수=1: 2590개
총등장수=2: 734개

총등장수 하위 30개:


,keyword,제품수,IP수,총등장수,제품명_예시
0,12곡,1,0,1,CJ)햇반12곡잡곡밥210g
1,10+1,1,0,1,양반들기름김세트
2,0106 : 단품할인,0,1,1,
3,(입력 상품명이 없습니다. 처리할 상품명을 제공해 주세요.),1,0,1,총동원)통그릴참치핫도그
4,(사용자 입력 대기 중),1,0,1,총동원)그릴리직화소시지파스타
5,스탠딩저니,1,0,1,나라)몬텔레나스탠딩저니나파샤도네이750ml
6,스키즈,1,0,1,롯데)아몬드8갑(스키즈25년)
7,스크류,1,0,1,M_직납)빅구슬스크류
8,스쿼시,1,0,1,후지야레몬스쿼시
9,스콧베이스,1,0,1,지비)스콧베이스피노누아750ml


## Phase 3.6. 키워드 정규화 (keyword_eda_final.csv 기반)

`keyword_eda_final.csv`의 `제거/검토` 컬럼을 읽어 세 가지 처리를 자동 적용합니다.

| `제거/검토` 값 | 처리 |
|---|---|
| `O` | 제거 (NOISE_REMOVE) |
| 단어 1개 | 1:1 대체 (KW_RENAME_MAP) |
| 단어 2개+ (쉼표 구분) | 1→N 분리 (KW_SPLIT_MAP) |

적용 순서: product_nodes → ip_nodes → kw_nodes 재구성

In [101]:
from collections import defaultdict

# ── keyword_eda_final.csv → NOISE / RENAME / SPLIT 맵 자동 구성 ────
kw_review  = pd.read_csv(HIN_DIR / "keyword_eda_final.csv", encoding="utf-8-sig")
review_col = kw_review.columns[-1]   # '제거/검토'

NOISE_REMOVE  = set()
KW_RENAME_MAP = {}   # 1:1 대체
KW_SPLIT_MAP  = {}   # 1→N 분리

for _, row in kw_review.iterrows():
    val = row[review_col]
    kw  = str(row["keyword"]).strip()
    if pd.isna(val) or str(val).strip() == "":
        continue
    val = str(val).strip()
    if val == "O":
        NOISE_REMOVE.add(kw)
    else:
        parts = [p.strip() for p in val.split(",") if p.strip()]
        if len(parts) == 1:
            KW_RENAME_MAP[kw] = parts[0]
        else:
            KW_SPLIT_MAP[kw] = parts

print(f"정규화 맵:")
print(f"  NOISE_REMOVE  : {len(NOISE_REMOVE):,}개  (O → 제거)")
print(f"  KW_RENAME_MAP : {len(KW_RENAME_MAP):,}쌍  (1:1 대체)")
print(f"  KW_SPLIT_MAP  : {len(KW_SPLIT_MAP):,}쌍  (1→N 분리)")

# ── apply_norm: 제거 + 1:1 대체 + 1→N 분리 + 중복 제거 ─────────
def apply_norm(kw_list):
    if not isinstance(kw_list, (list, np.ndarray)):
        return []
    seen, result = set(), []
    for k in kw_list:
        k = str(k).strip()
        if not k or k in NOISE_REMOVE:
            continue
        if k in KW_SPLIT_MAP:
            for part in KW_SPLIT_MAP[k]:
                if part not in NOISE_REMOVE and part not in seen:
                    seen.add(part); result.append(part)
        else:
            mapped = KW_RENAME_MAP.get(k, k)
            if mapped not in NOISE_REMOVE and mapped not in seen:
                seen.add(mapped); result.append(mapped)
    return result

# ── 정규화 전 vocab 크기 기록 ────────────────────────────────────
before_prod_vocab = len({k for lst in product_nodes["키워드_final"] for k in lst})
before_ip_vocab   = len({k for lst in ip_nodes["키워드_final"] for k in lst})

# ── product_nodes + ip_nodes에 정규화 적용 ──────────────────────
product_nodes["키워드_final"] = product_nodes["키워드_final"].apply(apply_norm)
ip_nodes["키워드_final"]      = ip_nodes["키워드_final"].apply(apply_norm)

# ── 정규화된 product_nodes 저장 (Phase 9 패치가 정규화 버전 기준으로 실행되도록)
product_nodes.to_parquet(HIN_DIR / "product_nodes.parquet", index=False)

# ── kw_nodes 재구성 (split 결과로 새로 생긴 키워드 포함) ─────────
prod_kw_vocab2 = {k for lst in product_nodes["키워드_final"] for k in lst}
ip_kw_vocab2   = {k for lst in ip_nodes["키워드_final"] for k in lst}
all_kw_vocab2  = prod_kw_vocab2 | ip_kw_vocab2

kw_nodes = pd.DataFrame({"keyword": sorted(all_kw_vocab2)})
kw_nodes["is_trend_keyword"] = kw_nodes["keyword"].isin(trend_kw_set)
kw_nodes["추출_속성"] = kw_nodes["keyword"].map(trend_attr).apply(
    lambda v: v if isinstance(v, list) else []
)
kw_nodes = kw_nodes.merge(kw_first_date, on="keyword", how="left")
kw_nodes.loc[~kw_nodes["is_trend_keyword"], "인스타_첫_등장일"] = None
kw_nodes.to_parquet(HIN_DIR / "keyword_nodes.parquet", index=False)

# valid_kws_final 갱신
valid_kws_final = set(kw_nodes["keyword"])

print()
print(f"제품 키워드 vocab : {before_prod_vocab:,}개 → {len(prod_kw_vocab2):,}개")
print(f"IP 키워드 vocab   : {before_ip_vocab:,}개 → {len(ip_kw_vocab2):,}개")
print(f"전체 합집합       : {len(all_kw_vocab):,}개 → {len(all_kw_vocab2):,}개")
print(f"keyword_nodes     : {len(kw_nodes):,}개 (트렌드: {kw_nodes['is_trend_keyword'].sum():,}개)")
print(f"유효 키워드       : {len(valid_kws_final):,}개")

정규화 맵:
  NOISE_REMOVE  : 1,061개  (O → 제거)
  KW_RENAME_MAP : 536쌍  (1:1 대체)
  KW_SPLIT_MAP  : 194쌍  (1→N 분리)

제품 키워드 vocab : 4,548개 → 3,102개
IP 키워드 vocab   : 756개 → 607개
전체 합집합       : 4,940개 → 3,336개
keyword_nodes     : 3,336개 (트렌드: 370개)
유효 키워드       : 3,336개


## Phase 3.7. 정규화 후 키워드 수 분포

In [102]:
# ── 제품별 키워드 수 분포 ────────────────────────────────────────
prod_kw_cnt = product_nodes["키워드_final"].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)
product_nodes["키워드_개수"] = prod_kw_cnt

bins   = [0, 1, 3, 5, 10, 20, 50, 999]
labels = ["0", "1-2", "3-4", "5-9", "10-19", "20-49", "50+"]

prod_bin = pd.cut(prod_kw_cnt, bins=bins, labels=labels, right=False)
print("=== 제품별 키워드 수 분포 (정규화 후) ===")
print()
dist_tbl = (
    product_nodes.assign(_bin=prod_bin)
    .groupby("편의점명")["_bin"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=labels)
)
print(dist_tbl.to_string())
print()
print(f"제품 전체 통계:")
print(f"  키워드 0개  : {(prod_kw_cnt == 0).sum():,}개")
print(f"  키워드 1-4개: {((prod_kw_cnt >= 1) & (prod_kw_cnt < 5)).sum():,}개")
print(f"  키워드 5+개 : {(prod_kw_cnt >= 5).sum():,}개")
print(f"  평균 키워드 : {prod_kw_cnt[prod_kw_cnt > 0].mean():.1f}개 (키워드 있는 제품 기준)")
print()

# ── IP별 키워드 수 분포 ──────────────────────────────────────────
ip_kw_cnt = ip_nodes["키워드_final"].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)
print("=== IP별 키워드 수 분포 (정규화 후) ===")
print()
ip_bin = pd.cut(ip_kw_cnt, bins=bins, labels=labels, right=False)
print(ip_bin.value_counts().sort_index().to_string())
print()
print(f"IP 전체 통계:")
print(f"  키워드 0개  : {(ip_kw_cnt == 0).sum():,}개")
print(f"  키워드 1-4개: {((ip_kw_cnt >= 1) & (ip_kw_cnt < 5)).sum():,}개")
print(f"  키워드 5+개 : {(ip_kw_cnt >= 5).sum():,}개")
print(f"  평균 키워드 : {ip_kw_cnt[ip_kw_cnt > 0].mean():.1f}개 (키워드 있는 IP 기준)")
print()

# ── 키워드 0개 제품 상세 (편의점별) ────────────────────────────
zero_kw = product_nodes[prod_kw_cnt == 0][["ITEM_CD", "ITEM_NM", "편의점명", "성공여부"]]
print(f"=== 키워드 0개 제품 ({len(zero_kw)}개) ===")
print(zero_kw.groupby("편의점명").size().to_string())
if len(zero_kw) > 0:
    print()
    display(zero_kw.head(20).reset_index(drop=True))

=== 제품별 키워드 수 분포 (정규화 후) ===

_bin    0  1-2  3-4   5-9  10-19  20-49  50+
편의점명                                        
CU     20    1   78   814    249      0    0
GS25   18    1   82   737    165      0    0
세븐일레븐  73   29  220  1693    978      3    0

제품 전체 통계:
  키워드 0개  : 111개
  키워드 1-4개: 411개
  키워드 5+개 : 4,639개
  평균 키워드 : 8.0개 (키워드 있는 제품 기준)

=== IP별 키워드 수 분포 (정규화 후) ===

키워드_final
0          0
1-2        9
3-4       52
5-9      203
10-19     12
20-49      0
50+        0

IP 전체 통계:
  키워드 0개  : 0개
  키워드 1-4개: 61개
  키워드 5+개 : 215개
  평균 키워드 : 5.9개 (키워드 있는 IP 기준)

=== 키워드 0개 제품 (111개) ===
편의점명
CU       20
GS25     18
세븐일레븐    73



,ITEM_CD,ITEM_NM,편의점명,성공여부
0,21778,롯데)리얼소프트에그샌드,세븐일레븐,실패
1,191602,우드)와인오프너(스크류),세븐일레븐,실패
2,202504,정관장)활력28리부스트100ml,세븐일레븐,실패
3,170145,오츠카)컨피던스230ml,세븐일레븐,실패
4,202456,종근당)산에는삼100ml 10입,세븐일레븐,실패
5,601437,★미루)원형부케,세븐일레븐,실패
6,191097,나라)산다라3본입패키지,세븐일레븐,실패
7,122273,롯데)칸타타여과지40매,세븐일레븐,실패
8,222510,프레시팜)한입삼겹살500g,세븐일레븐,실패
9,601432,★미루)물망초,세븐일레븐,실패


## Phase 4. 엣지: 제품 → 키워드

In [103]:
# Phase 3.6에서 product_nodes 키워드_final에 정규화 적용 완료
# 여기서는 valid_kws_final 기준으로 엣지 생성
valid_kws = valid_kws_final

prod_kw_edges = (
    product_nodes[["ITEM_CD", "키워드_final"]]
    .explode("키워드_final")
    .rename(columns={"키워드_final": "keyword"})
    .dropna(subset=["keyword"])
    .assign(keyword=lambda d: d["keyword"].astype(str))
    .query("keyword in @valid_kws")
    .drop_duplicates()
    .reset_index(drop=True)
)

prod_kw_edges.to_parquet(HIN_DIR / "product_keyword_edges.parquet", index=False)
print(f"product_keyword_edges: {len(prod_kw_edges)}행 → saved")
print(f"커버리지: {prod_kw_edges['ITEM_CD'].nunique()}개 제품 / {prod_kw_edges['keyword'].nunique()}개 키워드")

product_keyword_edges: 40321행 → saved
커버리지: 5032개 제품 / 3102개 키워드


## Phase 4-Fill — under-tagged 제품 키워드 보충

`keyword_fill_edges.parquet`가 존재하면 `prod_kw_edges`에 병합합니다.
파일 없으면 건너뜁니다 — 먼저 `python src/data_builder/batch_keyword_fill.py` 실행.

In [104]:
# keyword-fill
# under-tagged 제품 키워드 보충 엣지 병합
FILL_PATH = PROC / "keyword_fill_edges.parquet"

if FILL_PATH.exists():
    fill_edges = pd.read_parquet(FILL_PATH)
    fill_edges = fill_edges[fill_edges["keyword"].isin(valid_kws)]
    valid_prods_set = set(product_nodes["ITEM_CD"])
    fill_edges = fill_edges[fill_edges["ITEM_CD"].isin(valid_prods_set)]

    before = len(prod_kw_edges)
    prod_kw_edges = (
        pd.concat([prod_kw_edges, fill_edges[["ITEM_CD", "keyword"]]], ignore_index=True)
        .drop_duplicates(["ITEM_CD", "keyword"])
        .reset_index(drop=True)
    )
    prod_kw_edges.to_parquet(HIN_DIR / "product_keyword_edges.parquet", index=False)

    added = len(prod_kw_edges) - before
    n_prods = fill_edges["ITEM_CD"].nunique()
    print(f"keyword-fill: +{added}개 엣지 추가 ({n_prods}개 제품 커버)")
    print(f"product_keyword_edges: {before} → {len(prod_kw_edges)}행")
else:
    print(f"[SKIP] {FILL_PATH} 없음 — batch_keyword_fill.py 먼저 실행")

keyword-fill: +1014개 엣지 추가 (569개 제품 커버)
product_keyword_edges: 40321 → 41335행


## Phase 5. 엣지: IP → 키워드

In [105]:
ip_kw_edges = (
    ip_nodes[["ip_name", "키워드_final"]]
    .explode("키워드_final")
    .rename(columns={"키워드_final": "keyword"})
    .dropna(subset=["keyword"])
    .assign(keyword = lambda d: d["keyword"].astype(str))
    .query("keyword in @valid_kws")
    .drop_duplicates()
    .reset_index(drop=True)
)

ip_kw_edges.to_parquet(HIN_DIR / "ip_keyword_edges.parquet", index=False)
print(f"ip_keyword_edges: {len(ip_kw_edges)}행 → saved")

ip_keyword_edges: 1632행 → saved


## Phase 6. 엣지: 트렌드키워드 → 속성키워드

In [106]:
trend_kw_edges = (
    df_trend[["트렌드_키워드", "추출_속성_final"]]
    .rename(columns={"트렌드_키워드": "src_keyword", "추출_속성_final": "attr_list"})
    .explode("attr_list")
    .rename(columns={"attr_list": "tgt_keyword"})
    .dropna(subset=["tgt_keyword"])
    .assign(
        src_keyword = lambda d: d["src_keyword"].astype(str),
        tgt_keyword = lambda d: d["tgt_keyword"].astype(str),
    )
    .query("src_keyword in @valid_kws and tgt_keyword in @valid_kws")
    .drop_duplicates()
    .reset_index(drop=True)
)

trend_kw_edges.to_parquet(HIN_DIR / "trend_keyword_edges.parquet", index=False)
print(f"trend_keyword_edges: {len(trend_kw_edges)}행 → saved")

trend_keyword_edges: 2303행 → saved


## Phase 7. 엣지: 제품 → IP

In [107]:
valid_ips   = set(ip_nodes["ip_name"])
valid_prods = set(product_nodes["ITEM_CD"])

# ── 소스 1: product_ip_mapping.xlsx Sheet1 (제품_IP_매핑) ─────────
sheet1 = xl_ip.parse("제품_IP_매핑")

# Sheet1 제품명 → ITEM_CD 매핑 (pfk 기준)
nm_to_cd = (
    df_pfk.drop_duplicates("제품명")
    .set_index("제품명")["ITEM_CD"]
    .to_dict()
)
# 편의점명 prefix 포함 합성 ID도 커버
def resolve_item_cd(nm):
    if nm in nm_to_cd:
        return nm_to_cd[nm]
    # CU_{nm}, GS25_{nm}, 세븐일레븐_{nm} 시도
    for pfx in ["CU_", "GS25_", "세븐일레븐_"]:
        key = pfx + nm
        if key in nm_to_cd:
            return nm_to_cd[key]
    return None

sheet1_edges = (
    sheet1[["제품명", "매칭_IP"]].dropna()
    .rename(columns={"매칭_IP": "ip_name"})
    .assign(ITEM_CD=lambda d: d["제품명"].apply(resolve_item_cd))
    .dropna(subset=["ITEM_CD"])
    [["ITEM_CD", "ip_name"]]
)

# ── 소스 2: pfk의 IP 컬럼 ────────────────────────────────────────
pfk_ip_col = df_pfk[df_pfk["IP"].notna()][["ITEM_CD", "IP"]].copy()
pfk_ip_col = (
    pfk_ip_col
    .assign(ip_name=pfk_ip_col["IP"].str.split(","))
    .explode("ip_name")
    .assign(ip_name=lambda d: d["ip_name"].str.strip())
    .dropna(subset=["ip_name"])
    .query("ip_name != ''")
    [["ITEM_CD", "ip_name"]]
)

# ── 합집합 ───────────────────────────────────────────────────────
prod_ip_edges = (
    pd.concat([sheet1_edges, pfk_ip_col], ignore_index=True)
    .query("ITEM_CD in @valid_prods and ip_name in @valid_ips")
    .drop_duplicates(["ITEM_CD", "ip_name"])
    .reset_index(drop=True)
)

prod_ip_edges.to_parquet(HIN_DIR / "product_ip_edges.parquet", index=False)
print(f"product_ip_edges: {len(prod_ip_edges)}행 → saved")
print(f"  소스: Sheet1={len(sheet1_edges)}행 / pfk.IP={len(pfk_ip_col)}행 → 합집합 {len(prod_ip_edges)}행")
print(f"  커버리지: {prod_ip_edges['ITEM_CD'].nunique()}개 제품 / {prod_ip_edges['ip_name'].nunique()}개 IP")

product_ip_edges: 1200행 → saved
  소스: Sheet1=1179행 / pfk.IP=1227행 → 합집합 1200행
  커버리지: 1038개 제품 / 209개 IP


## Phase 7.5. 프로모션 원핫 인코딩 → product_nodes 피처 추가

`product_promo_keywords.csv` 기반 18종 프로모션 유형을 원핫 벡터로 변환하여 `product_nodes`에 병합합니다.  
컬럼명: `promo_{코드}_{유형명}` (코드 있는 경우) / `promo_{유형명}` (1+1 등 묶음 행사)

In [108]:
import re

def make_promo_slug(p: str) -> str:
    """'0106 : 단품할인' → 'promo_0106_단품할인' / '1+1' → 'promo_1+1'"""
    m = re.match(r"(\d+)\s*:\s*(.+)", p.strip())
    if m:
        return f"promo_{m.group(1)}_{m.group(2).strip()}"
    return f"promo_{p.strip()}"

# ── 전체 프로모션 vocab (슬러그 매핑) ───────────────────────────
all_promos_raw = set()
for v in df_promo["프로모션_키워드"].dropna():
    for p in str(v).split(","):
        p = p.strip()
        if p:
            all_promos_raw.add(p)

slug_map = {p: make_promo_slug(p) for p in all_promos_raw}   # raw → slug
all_slugs = sorted(slug_map.values())

print(f"프로모션 유형 {len(all_slugs)}종:")
for raw, slug in sorted(slug_map.items()):
    print(f"  {raw:<25} → {slug}")

# ── 제품별 원핫 피벗 ─────────────────────────────────────────────
promo_long = (
    df_promo[["ITEM_CD", "프로모션_키워드"]]
    .assign(
        promo_list=lambda d: d["프로모션_키워드"].apply(
            lambda v: [slug_map[p.strip()] for p in str(v).split(",")
                       if p.strip() in slug_map]
        )
    )
    .explode("promo_list")
    .rename(columns={"promo_list": "slug"})
    .dropna(subset=["slug"])
    .assign(val=1)
)

promo_onehot = (
    promo_long
    .pivot_table(index="ITEM_CD", columns="slug", values="val", fill_value=0)
    .reindex(columns=all_slugs, fill_value=0)
    .reset_index()
    .astype({s: "int8" for s in all_slugs})
)

# ── product_nodes에 병합 후 저장 ─────────────────────────────────
product_nodes = product_nodes.merge(promo_onehot, on="ITEM_CD", how="left")
for s in all_slugs:
    product_nodes[s] = product_nodes[s].fillna(0).astype("int8")

product_nodes.to_parquet(HIN_DIR / "product_nodes.parquet", index=False)

n_with_promo = (product_nodes[all_slugs].sum(axis=1) > 0).sum()
print()
print(f"프로모션 있는 제품: {n_with_promo:,}개 / {len(product_nodes):,}개")
print()
print("유형별 제품 수:")
for s in all_slugs:
    n = product_nodes[s].sum()
    if n > 0:
        print(f"  {n:4d}개  {s}")

프로모션 유형 18종:
  0101 : 번들증정               → promo_0101_번들증정
  0102 : 콤보증정               → promo_0102_콤보증정
  0103 : 번들할인               → promo_0103_번들할인
  0104 : 콤보할인               → promo_0104_콤보할인
  0106 : 단품할인               → promo_0106_단품할인
  0107 : 묶음할인(구간)           → promo_0107_묶음할인(구간)
  0201 : 번들증정               → promo_0201_번들증정
  0203 : 번들할인               → promo_0203_번들할인
  0205 : 장바구니할인             → promo_0205_장바구니할인
  0301 : 구독행사               → promo_0301_구독행사
  1+1                       → promo_1+1
  10+1                      → promo_10+1
  2+1                       → promo_2+1
  2+2                       → promo_2+2
  3+1                       → promo_3+1
  5+1                       → promo_5+1
  6+1                       → promo_6+1
  7+1                       → promo_7+1

프로모션 있는 제품: 1,481개 / 5,161개

유형별 제품 수:
   402개  promo_0101_번들증정
    64개  promo_0102_콤보증정
    52개  promo_0103_번들할인
   250개  promo_0104_콤보할인
   459개  promo_0106_단품할인
   106개  promo_0107_묶음할인(구간)
     3

## Phase 8. 통계 요약

In [109]:
print("=" * 55)
print("HIN 그래프 요약")
print("=" * 55)
print("[노드]")
print(f"  제품      : {len(product_nodes):,}개")
print(f"  IP        : {len(ip_nodes):,}개")
print(f"  키워드    : {len(kw_nodes):,}개  (트렌드: {kw_nodes['is_trend_keyword'].sum():,}개)")
print()
print("[엣지]")
print(f"  제품→키워드    : {len(prod_kw_edges):,}행")
print(f"  IP→키워드      : {len(ip_kw_edges):,}행")
print(f"  트렌드→키워드  : {len(trend_kw_edges):,}행")
print(f"  제품→IP        : {len(prod_ip_edges):,}행")
print()
print("[커버리지]")
print(f"  키워드 있는 제품  : {(product_nodes['키워드_final'].apply(len) > 0).sum():,}개 / {len(product_nodes):,}개")
print(f"  IP 연결 제품      : {prod_ip_edges['ITEM_CD'].nunique():,}개")
print(f"  프로모션 있는 제품: {(product_nodes[all_slugs].sum(axis=1) > 0).sum():,}개")
print()
print(f"[정규화]  제거 {len(NOISE_REMOVE)}개 / 병합 {len(KW_RENAME_MAP)}쌍 / 분리 {len(KW_SPLIT_MAP)}쌍")
print()
print(f"저장 경로: {HIN_DIR}")
print()
print("[출력 파일]")
for f in sorted(HIN_DIR.glob("*.parquet")):
    print(f"  {f.name}")

HIN 그래프 요약
[노드]
  제품      : 5,161개
  IP        : 276개
  키워드    : 3,336개  (트렌드: 370개)

[엣지]
  제품→키워드    : 41,335행
  IP→키워드      : 1,632행
  트렌드→키워드  : 2,303행
  제품→IP        : 1,200행

[커버리지]
  키워드 있는 제품  : 5,050개 / 5,161개
  IP 연결 제품      : 1,038개
  프로모션 있는 제품: 1,481개

[정규화]  제거 1061개 / 병합 536쌍 / 분리 194쌍

저장 경로: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\hin

[출력 파일]
  ip_keyword_edges.parquet
  ip_nodes.parquet
  keyword_nodes.parquet
  product_ip_edges.parquet
  product_keyword_edges.parquet
  product_nodes.parquet
  trend_keyword_edges.parquet
